In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from OptimalObservableHelper import OptimalObservableHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x86ecf10
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8a4c190


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# n_threads = 6
no_rvec = True
# write_outputs = False
write_outputs = True
dataset_path = "data/datasets/reweighted/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted/signal-only.json"
friend_dataset_path = "data/datasets/truth-reweighted-hel/signal-only.json"


In [4]:
ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)
friend_dataset = Dataset.from_json(friend_dataset_path)

In [6]:
analysis = OptimalObservableHelper(dataset, friend_datasets=[friend_dataset])

missing friend for sample: 4f_sw_sl_eLpL_bkg
missing friend for sample: 4f_sw_sl_eLpR_bkg
missing friend for sample: 4f_sw_sl_eRpR_bkg
missing friend for sample: 4f_sw_sl_eRpL_bkg
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xc72b710


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
slope_comp_g_orders = list(range(3, 17))

oo_configs = [
    "g1z_pos_1em08",
    "ka_pos_1em08",
    "la_pos_1em08",
    ]
oo_config_g_val = [f"g1z_pos_1em{x:02}" for x in slope_comp_g_orders]
oo_config_g_val += [f"ka_pos_1em{x:02}" for x in slope_comp_g_orders]
oo_config_g_val += [f"la_pos_1em{x:02}" for x in slope_comp_g_orders]
# oo_names = [f"O_{c}" for c in oo_configs]
oo_names = {
    # "mlvec_reco_oo": analysis.define_optimal_observables("mlvec_O", ["mlvec_reco_sqme", "wj_mlvec_reco_sqme"], oo_configs, categories=signal_category),
    # "reco_oo": analysis.define_optimal_observables("O", ["reco_sqme", "wj_reco_sqme"], oo_configs, categories=signal_category),
    # "reco_jm_oo": analysis.define_optimal_observables("jm_O", ["reco_sqme"], oo_configs, categories=signal_category),
    # "clean_reco_oo": analysis.define_optimal_observables("clean_O", ["clean_reco_sqme", "wj_clean_reco_sqme"], oo_configs, categories=signal_category),
    # "clean_reco_jm_oo": analysis.define_optimal_observables("clean_jm_O", ["clean_reco_sqme"], oo_configs, categories=signal_category),
    # "cheat_clean_reco_oo": analysis.define_optimal_observables("cheat_clean_O", ["cheat_clean_reco_sqme", "wj_cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    # "cheat_clean_reco_jm_oo": analysis.define_optimal_observables("cheat_clean_jm_O", ["cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    "mc_oo": analysis.define_optimal_observables("mc_O", ["mc_sqme"], oo_config_g_val, categories=signal_category),
    # "nomb_mc_oo": analysis.define_optimal_observables("nomb_mc_O", ["nomb_mc_sqme"], oo_configs, categories=signal_category),
    # "nomb_mc_rlep_oo": analysis.define_optimal_observables("nomb_mc_rlep_O", ["nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
    # "av_mc_oo": analysis.define_optimal_observables("av_mc_O", ["mc_sqme", "wj_mc_sqme"], oo_configs, categories=signal_category),
    # "av_nomb_mc_oo": analysis.define_optimal_observables("av_nomb_mc_O", ["nomb_mc_sqme", "wj_nomb_mc_sqme"], oo_configs, categories=signal_category),
    # "av_nomb_mc_rlep_oo": analysis.define_optimal_observables("av_nomb_mc_rlep_O", ["nomb_mc_rlep_sqme", "wj_nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
}

In [9]:
for names in oo_names.values():
    for name in names:
        # create dummy oo where they don't exist yet
        analysis.define_only_on(["4f_sl_bkg"], name, "0.")
# TODO: re-consider what is correct here...
analysis.add_filter("&&".join([f"std::isfinite({oo})" for oo_name in oo_names.values() for oo in oo_name]), "finite OO")
# analysis.add_filter("&&".join([f"abs({oo}) <= 5" for oo in oo_names["reco_oo"]]), "abs(OO) <= 5")
# analysis.add_filter("(iso_lep_lvec + nu_lvec).M() > 0.", "physical nu")
# analysis.add_filter("(iso_lep_lvec + clean_nu_lvec).M() > 0.", "physical clean_nu")
# analysis.add_filter("nu_lvec.E() > 0.", "physical nu")
# analysis.add_filter("clean_nu_lvec.E() > 0.", "physical clean nu")
analysis.book_reports()

In [10]:
analysis.book_histogram_1D("mc_O_g1z_pos_1em08", "mc_O_g1z_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)
analysis.book_histogram_1D("mc_O_ka_pos_1em08", "mc_O_ka_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)
analysis.book_histogram_1D("mc_O_la_pos_1em08", "mc_O_la_pos_1em08", ("", "", 250, -2.5, 2.5), categories=signal_category)

In [11]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0
  },
"variations": [
    0.0025,
    -0.0025,
    0.002,
    -0.002,
    0.0015,
    -0.0015,
    0.001,
    -0.001,
    7.5e-04,
    -7.5e-04,
    5e-04,
    -5e-04,
    2.5e-04,
    -2.5e-04,
    1e-04,
    -1e-04,
    1e-05,
    1e-06,
    1e-07,
    1e-08,
    1e-09,
    1e-10,
    1e-11,
    1e-12,
    1e-13,
    1e-14,
    1e-15,
    1e-16
  ]
}
""", mirror=False, combinations=False)
alt_config_names = list(alt_setup_handler.get_alt_setup().keys())

In [12]:
weight_names = analysis.book_weight_sums(["nominal"] + alt_config_names, categories=signal_category)
# weight_names = analysis.book_weight_sums(["nominal"] + [name for name in alt_config_names if "em03" in name] + [name for name in alt_config_names if "em04" in name], categories=signal_category)

In [13]:
for names in oo_names.values():
    analysis.define_weighted_oo(names, weight_names, categories=signal_category)
    analysis.book_oo_sums(names, weight_names, categories=signal_category)
    # analysis.book_oo_matrix(names, categories=signal_category)

In [14]:
pars = ["g1z", "ka", "la"]
for x in slope_comp_g_orders:
    analysis.book_oo_matrix([f"mc_O_{p}_pos_1em{x:02}" for p in pars])

In [15]:
%%time
analysis.run()

CPU times: user 58min 54s, sys: 25.7 s, total: 59min 20s
Wall time: 6min 19s


In [16]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
        10244400 (1e-03)           33239 (2e-02) All
        10244400 (1e-03)           33239 (2e-02) finite OO
                    1.00                    1.00 efficiency



In [17]:
print(alt_config_names)

['g1z_pos_3em03', 'ka_pos_3em03', 'la_pos_3em03', 'g1z_neg_3em03', 'ka_neg_3em03', 'la_neg_3em03', 'g1z_pos_2em03', 'ka_pos_2em03', 'la_pos_2em03', 'g1z_neg_2em03', 'ka_neg_2em03', 'la_neg_2em03', 'g1z_pos_1em03', 'ka_pos_1em03', 'la_pos_1em03', 'g1z_neg_1em03', 'ka_neg_1em03', 'la_neg_1em03', 'g1z_pos_8em04', 'ka_pos_8em04', 'la_pos_8em04', 'g1z_neg_8em04', 'ka_neg_8em04', 'la_neg_8em04', 'g1z_pos_5em04', 'ka_pos_5em04', 'la_pos_5em04', 'g1z_neg_5em04', 'ka_neg_5em04', 'la_neg_5em04', 'g1z_pos_3em04', 'ka_pos_3em04', 'la_pos_3em04', 'g1z_neg_3em04', 'ka_neg_3em04', 'la_neg_3em04', 'g1z_pos_1em04', 'ka_pos_1em04', 'la_pos_1em04', 'g1z_neg_1em04', 'ka_neg_1em04', 'la_neg_1em04', 'g1z_pos_1em05', 'ka_pos_1em05', 'la_pos_1em05', 'g1z_pos_1em06', 'ka_pos_1em06', 'la_pos_1em06', 'g1z_pos_1em07', 'ka_pos_1em07', 'la_pos_1em07', 'g1z_pos_1em08', 'ka_pos_1em08', 'la_pos_1em08', 'g1z_pos_1em09', 'ka_pos_1em09', 'la_pos_1em09', 'g1z_pos_1em10', 'ka_pos_1em10', 'la_pos_1em10', 'g1z_pos_1em11', 'k

In [18]:
# calculate means etc.
names = oo_names["mc_oo"]
# names = oo_names["mlvec_reco_oo"]

oo_means = analysis.calc_oo_means(names, weight_names, categories=signal_category)


In [19]:
print(names)

['mc_O_g1z_pos_1em03', 'mc_O_g1z_pos_1em04', 'mc_O_g1z_pos_1em05', 'mc_O_g1z_pos_1em06', 'mc_O_g1z_pos_1em07', 'mc_O_g1z_pos_1em08', 'mc_O_g1z_pos_1em09', 'mc_O_g1z_pos_1em10', 'mc_O_g1z_pos_1em11', 'mc_O_g1z_pos_1em12', 'mc_O_g1z_pos_1em13', 'mc_O_g1z_pos_1em14', 'mc_O_g1z_pos_1em15', 'mc_O_g1z_pos_1em16', 'mc_O_ka_pos_1em03', 'mc_O_ka_pos_1em04', 'mc_O_ka_pos_1em05', 'mc_O_ka_pos_1em06', 'mc_O_ka_pos_1em07', 'mc_O_ka_pos_1em08', 'mc_O_ka_pos_1em09', 'mc_O_ka_pos_1em10', 'mc_O_ka_pos_1em11', 'mc_O_ka_pos_1em12', 'mc_O_ka_pos_1em13', 'mc_O_ka_pos_1em14', 'mc_O_ka_pos_1em15', 'mc_O_ka_pos_1em16', 'mc_O_la_pos_1em03', 'mc_O_la_pos_1em04', 'mc_O_la_pos_1em05', 'mc_O_la_pos_1em06', 'mc_O_la_pos_1em07', 'mc_O_la_pos_1em08', 'mc_O_la_pos_1em09', 'mc_O_la_pos_1em10', 'mc_O_la_pos_1em11', 'mc_O_la_pos_1em12', 'mc_O_la_pos_1em13', 'mc_O_la_pos_1em14', 'mc_O_la_pos_1em15', 'mc_O_la_pos_1em16']


In [20]:
print(oo_means.keys())

dict_keys(['mc_O_g1z_pos_1em03_weight_nominal', 'mc_O_g1z_pos_1em04_weight_nominal', 'mc_O_g1z_pos_1em05_weight_nominal', 'mc_O_g1z_pos_1em06_weight_nominal', 'mc_O_g1z_pos_1em07_weight_nominal', 'mc_O_g1z_pos_1em08_weight_nominal', 'mc_O_g1z_pos_1em09_weight_nominal', 'mc_O_g1z_pos_1em10_weight_nominal', 'mc_O_g1z_pos_1em11_weight_nominal', 'mc_O_g1z_pos_1em12_weight_nominal', 'mc_O_g1z_pos_1em13_weight_nominal', 'mc_O_g1z_pos_1em14_weight_nominal', 'mc_O_g1z_pos_1em15_weight_nominal', 'mc_O_g1z_pos_1em16_weight_nominal', 'mc_O_ka_pos_1em03_weight_nominal', 'mc_O_ka_pos_1em04_weight_nominal', 'mc_O_ka_pos_1em05_weight_nominal', 'mc_O_ka_pos_1em06_weight_nominal', 'mc_O_ka_pos_1em07_weight_nominal', 'mc_O_ka_pos_1em08_weight_nominal', 'mc_O_ka_pos_1em09_weight_nominal', 'mc_O_ka_pos_1em10_weight_nominal', 'mc_O_ka_pos_1em11_weight_nominal', 'mc_O_ka_pos_1em12_weight_nominal', 'mc_O_ka_pos_1em13_weight_nominal', 'mc_O_ka_pos_1em14_weight_nominal', 'mc_O_ka_pos_1em15_weight_nominal', 'mc

In [21]:
oo_mat_g_val_graphs = []

from itertools import combinations_with_replacement
for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
    oo_mat_g_val_graphs.append(ROOT.TGraph())
    # could set title already here as I have p1 and p2 available

for x in slope_comp_g_orders:
    mat = analysis.get_oo_matrix([f"mc_O_{p}_pos_1em{x:02}" for p in pars])
    for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
        oo_mat_g_val_graphs[i].AddPoint(x, mat[i])

oo_mat_val_canvs = []
for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
    c = ROOT.TCanvas()
    oo_mat_g_val_graphs[i].Draw("alp")
    c.Draw()
    oo_mat_val_canvs.append(c)


In [22]:
mean_g_val_graphs = {}

for par in pars:
    graph = ROOT.TGraph()
    for x in slope_comp_g_orders:
        mean = oo_means[f"mc_O_{par}_pos_1em{x:02}_weight_nominal"]
        graph.AddPoint(x, mean)

    mean_g_val_graphs[par] = graph

mgval_max = {
    "g1z": -0.10255,
    "ka": -0.06895,
    "la": 0.092,
}
mgval_min = {
    "g1z": -0.1044,
    "ka": -0.06938,
    "la": 0.09025,
}
mgval_canvs = {}
for k, graph in mean_g_val_graphs.items():
    c = ROOT.TCanvas()
    graph.SetTitle(f";-log_{{10}}(#Delta {k}); <O_{{{k}}}>")
    graph.SetMinimum(mgval_min[k])
    graph.SetMaximum(mgval_max[k])
    graph.Draw("alp")
    c.Draw()
    mgval_canvs[k] = c


In [23]:
oo_slopes = analysis.get_slopes(names, pars, oo_means, g=1e-4)

In [24]:
print(oo_slopes.keys())
print(names)

dict_keys(['mc_O_g1z_pos_1em03_g1z', 'mc_O_g1z_pos_1em03_ka', 'mc_O_g1z_pos_1em03_la', 'mc_O_g1z_pos_1em04_g1z', 'mc_O_g1z_pos_1em04_ka', 'mc_O_g1z_pos_1em04_la', 'mc_O_g1z_pos_1em05_g1z', 'mc_O_g1z_pos_1em05_ka', 'mc_O_g1z_pos_1em05_la', 'mc_O_g1z_pos_1em06_g1z', 'mc_O_g1z_pos_1em06_ka', 'mc_O_g1z_pos_1em06_la', 'mc_O_g1z_pos_1em07_g1z', 'mc_O_g1z_pos_1em07_ka', 'mc_O_g1z_pos_1em07_la', 'mc_O_g1z_pos_1em08_g1z', 'mc_O_g1z_pos_1em08_ka', 'mc_O_g1z_pos_1em08_la', 'mc_O_g1z_pos_1em09_g1z', 'mc_O_g1z_pos_1em09_ka', 'mc_O_g1z_pos_1em09_la', 'mc_O_g1z_pos_1em10_g1z', 'mc_O_g1z_pos_1em10_ka', 'mc_O_g1z_pos_1em10_la', 'mc_O_g1z_pos_1em11_g1z', 'mc_O_g1z_pos_1em11_ka', 'mc_O_g1z_pos_1em11_la', 'mc_O_g1z_pos_1em12_g1z', 'mc_O_g1z_pos_1em12_ka', 'mc_O_g1z_pos_1em12_la', 'mc_O_g1z_pos_1em13_g1z', 'mc_O_g1z_pos_1em13_ka', 'mc_O_g1z_pos_1em13_la', 'mc_O_g1z_pos_1em14_g1z', 'mc_O_g1z_pos_1em14_ka', 'mc_O_g1z_pos_1em14_la', 'mc_O_g1z_pos_1em15_g1z', 'mc_O_g1z_pos_1em15_ka', 'mc_O_g1z_pos_1em15_la', '

In [25]:
# oo_suffix = "_pos_1em08"

# slope_val_graphs = {k.replace(oo_suffix, ""): ROOT.TGraph() for k in oo_slopes}

# for x in slope_comp_g_orders:
#     g = 10**(-x)
#     # print(g)
#     slopes = analysis.get_slopes(names, pars, oo_means, g=g)
#     for k, s in slopes.items():
#         key = k.replace(oo_suffix, "")
#         slope_val_graphs[key].AddPoint(x, s)

# slope_val_canvs = {}
# for k, g in slope_val_graphs.items():
#     c = ROOT.TCanvas()
#     g.SetTitle(k)
#     g.Draw("alp")
#     c.Draw()
#     slope_val_canvs[k] = c

In [26]:
# x_points = [-2e-3, -1.5e-3, -1e-3, -5e-4, 5e-4, 1e-3, 1.5e-3, 2e-3]
x_points = [-1.5e-3, -1e-3, -5e-4, -2.5e-4, -1e-4, 1e-4, 2.5e-4, 5e-4, 1e-3, 1.5e-3]
oo_graphs = analysis.make_slope_graphs([f"mc_O_{p}_pos_1em08" for p in pars], pars, x_points, oo_means)

In [70]:
canvases = {}
f_slopes = {}
r_graphs = {}
for k, g in oo_graphs.items():
    print(k)
    x_unit = k.split("_")[-1]
    o_unit = k.removesuffix(f"_pos_1em08_{x_unit}").split("_")[-1]
    c = ROOT.TCanvas()
    left_margin = 0.23
    # left_margin = ROOT.gStyle.GetPadLeftMargin()
    right_margin = 0.05
    bottom_ratio = 0.5
    top_ratio = 1. - bottom_ratio
    c.DivideRatios(1, 2, [1.], [top_ratio, bottom_ratio])
    c.cd(1)
    c.GetPad(1).SetBottomMargin(0.)
    c.GetPad(1).SetLeftMargin(left_margin)
    c.GetPad(1).SetRightMargin(right_margin)
    g.SetTitle(f";;#frac{{#Delta E[O_{{{o_unit}}}] }}{{E_{{0}}[O_{{{o_unit}}}]}}[%]")
    g.Draw("alp")
    g.GetYaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / top_ratio))
    g.GetYaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * 0.85 * (1 / top_ratio))
    g.GetYaxis().SetTitleOffset(ROOT.gStyle.GetTitleOffset()* 1.6 * top_ratio)
    # f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x + 1.", -0.01, 0.01)
    # f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x", -0.01, 0.01)
    f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x * 100", -0.01, 0.01)
    f.Draw("same")
    # f.Draw()
    f_slopes[k] = f
    c.cd(2)
    c.GetPad(2).SetTopMargin(0.)
    c.GetPad(2).SetBottomMargin(0.175 * (1. / bottom_ratio))
    c.GetPad(2).SetLeftMargin(left_margin)
    c.GetPad(2).SetRightMargin(right_margin)
    r_graph = analysis.make_ratio_graph(g, f)
    r_graph.Draw("alp")
    r_graph.GetYaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / bottom_ratio))
    r_graph.GetXaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / bottom_ratio))
    r_graph.GetXaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * (1 / bottom_ratio))
    r_graph.GetYaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * 0.85 * (1 / top_ratio))
    r_graph.GetYaxis().SetTitleOffset(ROOT.gStyle.GetTitleOffset()* 1.6 * top_ratio)
    r_graph.SetTitle(f";{x_unit};difference [10^{{-6}}]")
    r_graphs[k] = r_graph
    c.Draw()
    canvases[k] = c

mc_O_g1z_pos_1em08_g1z
mc_O_g1z_pos_1em08_ka
mc_O_g1z_pos_1em08_la
mc_O_ka_pos_1em08_g1z
mc_O_ka_pos_1em08_ka
mc_O_ka_pos_1em08_la
mc_O_la_pos_1em08_g1z
mc_O_la_pos_1em08_ka
mc_O_la_pos_1em08_la
